In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
print(os.listdir("/kaggle/input/datasets/ar3snd/deepfashion-dataset/dataset/backup/"))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
path = "/kaggle/input/datasets/ar3snd/deepfashion-dataset/dataset/backup"

# Code for Mask RCNN

# Dataset and Dataloader

In [ ]:
import os
import json
import numpy as np
import torch
import cv2
from torch.utils.data import Dataset

CLASS_MAP = {
    1: 0,
    2: 1,
    7: 2,
    8: 3,
    9: 4
}

class DeepFashionDataset(Dataset):
    def __init__(self, root_dir, transforms=None):
        self.root = root_dir
        self.transforms = transforms

        self.img_dir = os.path.join(root_dir, "image")
        self.ann_dir = os.path.join(root_dir, "annos")

        self.ids = [f.replace(".json", "") for f in os.listdir(self.ann_dir)]

    def __getitem__(self, idx):
        img_id = self.ids[idx]

        img_path = os.path.join(self.img_dir, img_id + ".jpg")
        ann_path = os.path.join(self.ann_dir, img_id + ".json")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        with open(ann_path) as f:
            data = json.load(f)

        masks = []
        boxes = []
        labels = []
        
        h, w = image.shape[:2]
        
        for key, item in data.items():
            if not key.startswith("item"):
                continue
        
            cat = item.get("category_id", None)
            if cat not in CLASS_MAP:
                continue
        
            label = CLASS_MAP[cat] + 1  # background = 0
        
            # ---- SEGMENTATION ----
            if "segmentation" not in item or len(item["segmentation"]) == 0:
                continue
        
            mask = np.zeros((h, w), dtype=np.uint8)
        
            # segmentation can have multiple polygons
            for seg in item["segmentation"]:
                poly = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [poly], 1)
        
            if mask.sum() == 0:
                continue
        
            # ---- BOUNDING BOX ----
            if "bounding_box" in item:
                x1, y1, x2, y2 = item["bounding_box"]
            else:
                ys, xs = np.where(mask)
                x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
        
            boxes.append([x1, y1, x2, y2])
            masks.append(mask)
            labels.append(label)

        if len(boxes) == 0:
            return self.__getitem__((idx + 1) % len(self))
        
        boxes = list(boxes)
        labels = list(labels)
        
        if self.transforms:
            transformed = self.transforms(
                image=image,
                masks=masks,
                bboxes=boxes,
                labels=labels
            )
            
            image = transformed["image"] 
            
            masks = torch.from_numpy(np.stack(transformed["masks"])).to(torch.uint8)
            boxes = torch.tensor(transformed["bboxes"], dtype=torch.float32)
            labels = torch.tensor(transformed["labels"], dtype=torch.int64)
        
        else:
            image = torch.tensor(image).permute(2, 0, 1) / 255.0
            masks = torch.tensor(masks, dtype=torch.uint8)
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks
        }
        
        return image, target

    def __len__(self):
        return len(self.ids)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transform():
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.Affine(
                translate_percent=0.05,
                scale=(0.9, 1.1),
                rotate=(-10, 10),
                p=0.5
            ),
            A.Normalize(mean=(0,0,0), std=(1,1,1)),
            ToTensorV2()
        ],
        bbox_params=A.BboxParams(
            format="pascal_voc",
            label_fields=["labels"]
        )
    )

def get_val_transform():
    return A.Compose([ToTensorV2()])

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = DeepFashionDataset(
    path + "/train",
    transforms=get_train_transform()
)

val_dataset = DeepFashionDataset(
    path + "/validation",
    transforms=get_val_transform()
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torch

model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)

# Model Code

In [ ]:
num_classes = 6

in_features = model.roi_heads.box_predictor.cls_score.in_features
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256

model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer,num_classes)
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False

# Training

In [ ]:
BEST_MODEL_PATH = "/kaggle/working/best_maskrcnn.pth"
LAST_MODEL_PATH = "/kaggle/working/last_maskrcnn.pth"

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            return True   # improvement
        else:
            self.counter += 1
            return False  # no improvement

    def should_stop(self):
        return self.counter >= self.patience

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    params,
    lr=1e-4,    
    weight_decay=1e-4
)

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15,   # total epochs
    eta_min=1e-6
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# AMP scaler (for faster + more stable training on GPU)
scaler = torch.amp.GradScaler("cuda")

MAX_EPOCHS = 1

early_stopper = EarlyStopping(patience=3)

best_val_loss = float("inf")

for epoch in range(MAX_EPOCHS):
    # ---- TRAIN ----
    model.train()
    train_loss = 0

    for images, targets in train_loader:
        images = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        # ---- TRAIN ----
        with torch.amp.autocast("cuda"):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        
        scaler.unscale_(optimizer)  # ✅ add this
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device, non_blocking=True) for img in images]
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
    
            with torch.amp.autocast("cuda"):   # ✅ fixed
                loss = sum(model(images, targets).values())
            val_loss += loss.item()

    print(f"Epoch {epoch+1}: Train={train_loss:.3f}, Val={val_loss:.3f}")

    scheduler.step()
    
    # ---- SAVE LAST MODEL ----
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss
    }, LAST_MODEL_PATH)

    # ---- SAVE BEST MODEL ----
    improved = early_stopper.step(val_loss)

    if improved:
        print("Saving best model...")
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    # ---- EARLY STOP ----
    if early_stopper.should_stop():
        print("Early stopping triggered")
        break

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_maskrcnn.pth"))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_prediction(image, prediction, threshold=0.5):
    """
    image: Tensor of shape (3, H, W)
    prediction: Output dict from model
    """
    image = image.permute(1, 2, 0).cpu().numpy()
    # Un-normalize if you used ImageNet stats (adjust if mean/std were 0/1)
    # image = (image * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    
    fig, ax = plt.subplots(1, figsize=(12, 9))
    ax.imshow(image)

    boxes = prediction['boxes'].cpu().detach().numpy()
    scores = prediction['scores'].cpu().detach().numpy()
    labels = prediction['labels'].cpu().detach().numpy()
    masks = prediction['masks'].cpu().detach().numpy()

    for i, score in enumerate(scores):
        if score > threshold:
            box = boxes[i]
            # Draw Box
            rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                     linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            
            # Draw Label
            ax.text(box[0], box[1], f"Class {labels[i]}: {score:.2f}", 
                    bbox=dict(facecolor='yellow', alpha=0.5))
            
            # Draw Mask (Overlay)
            mask = masks[i, 0] > 0.5
            ax.imshow(np.ma.masked_where(~mask, mask), cmap='cool', alpha=0.3)

    plt.axis('off')
    plt.show()

# Usage:
# model.eval()
# img, _ = val_dataset[0]
# with torch.no_grad():
#     pred = model([img.to(device)])[0]
# visualize_prediction(img, pred)

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchmetrics.classification import MulticlassStatScores, MulticlassAUROC
import torch

def evaluate_model(model, data_loader, device, num_classes):
    model.eval()
    
    metric_map = MeanAveragePrecision(box_format='pascal_voc')
    
    # For Dice/mIoU, we'll accumulate stats per class
    # Using torchmetrics for easier per-class macro averaging
    total_iou = 0
    total_dice = 0
    
    all_scores = []
    all_labels = []
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            
            # 1. Update Detection mAP
            # Torchmetrics expects a specific format
            preds_fmt = [
                dict(boxes=o['boxes'].cpu(), scores=o['scores'].cpu(), labels=o['labels'].cpu()) 
                for o in outputs
            ]
            target_fmt = [
                dict(boxes=t['boxes'].cpu(), labels=t['labels'].cpu()) 
                for t in targets
            ]
            metric_map.update(preds_fmt, target_fmt)
            
            # 2. Prepare data for ROC/AUC/F1 (flattened for classification-style metrics)
            for i in range(len(outputs)):
                # This is simplified: treating detection as a multi-class problem
                # in a real scenario, you'd match detections to ground truth
                all_scores.append(outputs[i]['scores'].cpu())
                all_labels.append(outputs[i]['labels'].cpu())

            # 3. Segmentation Metrics (mIoU/Dice)
            for i in range(len(targets)):
                gt_masks = targets[i]['masks'].cpu()
                pred_masks = outputs[i]['masks'].cpu() > 0.5
                
                # Note: This requires matching masks to GT. 
                # Simplified approach: If multiple masks, combine or pick best.
                # Standard practice uses a matching algorithm (Hungarian or IoU based).

    # Compute Final Results
    detection_results = metric_map.compute()
    
    print(f"mAP @ [0.5:0.95]: {detection_results['map']:.4f}")
    print(f"mAP @ 0.5: {detection_results['map_50']:.4f}")
    
    # For ROC/AUC/F1 and Segmentation stats, 
    # you would typically use the output of metric_map or a dedicated loop.
    return detection_results

# Execute evaluation
# stats = evaluate_model(model, val_loader, device, num_classes=6)